# Live Headline Index Polling

## Integration Guide: Pre-Aggregated Sentiment Index

This notebook demonstrates how to integrate the **Permutable AI Live Headline Index API** into a systematic trading workflow. The index endpoint returns sentiment that has already been aggregated into hourly buckets, making it easier to consume directly in quantitative strategies without client-side aggregation.

### How the Index Differs from the Raw Feed

| | Raw Feed (`/headlines/feed/live`) | Index (`/headlines/index/ticker/live`) |
|---|---|---|
| **Granularity** | One row per headline | One row per (ticker, hour, topic) |
| **Key metrics** | Per-headline `sentiment_score`, probabilities | `sentiment_sum`, `sentiment_abs_sum`, `headline_count`, `sentiment_std` |
| **Client work** | Must aggregate before use in strategy | Ready to use directly |
| **Best for** | Research, custom aggregations, NLP feature engineering | Systematic strategies, dashboards, live signals |

### What the Endpoint Returns

`GET /v1/headlines/index/ticker/live/{ticker}` returns the latest 2 hours of pre-aggregated sentiment index data. Each record maps to the `HeadlineIndex` schema:

| Field | Type | Description |
|---|---|---|
| `ticker` | `str` | Asset ticker (e.g. `BZ_COM`) |
| `publication_time` | `datetime` | Start or end of the hourly bucket |
| `topic_name` | `str` | Sentiment topic |
| `index_type` | `str` | `EXPLICIT` / `IMPLICIT` / `COMBINED` |
| `headline_count` | `int` | Number of headlines in this bucket |
| `sentiment_sum` | `float` | Sum of sentiment scores (net directional signal) |
| `sentiment_abs_sum` | `float` | Sum of absolute sentiment scores (total conviction) |
| `sentiment_std` | `float` | Std dev of sentiment scores (dispersion / disagreement) |

**Derived metric — Conviction Ratio:**

```
conviction_ratio = sentiment_sum / sentiment_abs_sum   ∈ [−1, +1]
```

A conviction ratio near +1 means all headlines in the bucket were bullish; near −1 means all were bearish. Values near 0 indicate mixed or neutral sentiment regardless of volume.

### Polling Architecture

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Every 15 minutes                                                       │
│                                                                         │
│  [API] ──► fetch_live_index(ticker)                                     │
│                  │                                                      │
│                  ▼                                                      │
│            upsert_index()  ──► [SQLite: headline_index]                 │
│                                       │                                 │
│                                       ▼                                 │
│                             load from DB ──► normalised strategy signal │
└─────────────────────────────────────────────────────────────────────────┘
```

**Upsert key:** `(ticker, publication_time, topic_name, index_type)`. The live endpoint always returns the current 2-hour window, so successive polls overlap. Upserting on this composite key keeps the database deduplicated without bookkeeping.

### Pipeline Steps

1. **Install** dependencies and configure API credentials
2. **Define** the fetch and upsert functions
3. **Dry-run** a single poll to validate connectivity and inspect the index schema
4. **Visualise** the initial hourly snapshot (sentiment bars, news flow)
5. **Run** the 15-minute polling loop
6. **Visualise** the accumulated live window (multi-ticker index, heatmap, conviction band)
7. **Derive** a normalised conviction-ratio signal for use in a systematic strategy

### Prerequisites

- A Permutable AI API key (available from your account dashboard)
- Python 3.9+ with the packages listed in the cell below

In [ ]:
# Install required packages — run this cell once, then restart the kernel if needed
%pip install requests pandas matplotlib seaborn

In [ ]:
import getpass
import sqlite3
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 5)})

In [ ]:
# ── API credentials ────────────────────────────────────────────────────────────
# The key is read via getpass so it is never displayed or stored in the notebook
API_KEY = getpass.getpass("Enter your Permutable AI API key: ")

# ── Endpoint configuration ─────────────────────────────────────────────────────
BASE_URL = "https://copilot-api.permutable.ai/v1"

# Edit this list to match the tickers included in your licence
TICKERS = ["BTC_CRY", "ETH_CRY", "BZ_COM", "EUR_IND"]

# Index type: COMBINED includes all headlines; EXPLICIT = direct mention; IMPLICIT = inferred
INDEX_TYPE = "COMBINED"  # EXPLICIT | IMPLICIT | COMBINED
TOPIC_PRESET = "ALL"  # topic preset name or "ALL"

# sparse=True returns only buckets that have headlines (recommended — avoids very large responses)
# align_to_period_end=True timestamps buckets at the close of each hour (e.g. 10:00 for 09:00–10:00)
SPARSE = True
ALIGN_TO_PERIOD_END = True

# ── Polling interval ───────────────────────────────────────────────────────────
POLL_INTERVAL_SECONDS = 15 * 60  # poll every 15 minutes

# ── Local storage ──────────────────────────────────────────────────────────────
DB_PATH = Path("headline_index.db")

print(f"Configured for {len(TICKERS)} tickers: {TICKERS}")
print(f"Index type       : {INDEX_TYPE}")
print(f"Polling interval : {POLL_INTERVAL_SECONDS // 60} minutes")
print(f"Local database   : {DB_PATH.resolve()}")

In [ ]:
def fetch_live_index(ticker: str) -> list[dict]:
    """
    Fetch the live headline sentiment index for a single ticker.

    Calls GET /v1/headlines/index/ticker/live/{ticker} and returns a flat list
    of record dicts matching the HeadlineIndex schema. A 'fetched_at' timestamp
    is appended so you can track when each poll retrieved each record.
    """
    url = f"{BASE_URL}/headlines/index/ticker/live/{ticker}"
    params = {
        "index_type": INDEX_TYPE,
        "topic_preset": TOPIC_PRESET,
        "sparse": str(SPARSE).lower(),
        "align_to_period_end": str(ALIGN_TO_PERIOD_END).lower(),
    }
    headers = {"x-api-key": API_KEY}

    response = requests.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()

    records = response.json()
    for r in records:
        r["fetched_at"] = datetime.now(timezone.utc).isoformat()
    return records


# Quick connectivity check
print(f"Testing connectivity for {TICKERS[0]}...")
test_records = fetch_live_index(TICKERS[0])
print(f"  OK — {len(test_records)} index records returned")

In [ ]:
def get_connection() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def setup_database() -> None:
    """Create the headline_index table and index if they do not already exist."""
    with get_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS headline_index (
                ticker              TEXT NOT NULL,
                publication_time    TEXT NOT NULL,
                topic_name          TEXT NOT NULL,
                index_type          TEXT NOT NULL,
                headline_count      INTEGER,
                sentiment_sum       REAL,
                sentiment_abs_sum   REAL,
                sentiment_std       REAL,
                fetched_at          TEXT,
                PRIMARY KEY (ticker, publication_time, topic_name, index_type)
            )
        """)
        conn.execute("""
            CREATE INDEX IF NOT EXISTS idx_hi_ticker_time
            ON headline_index (ticker, publication_time)
        """)
    print(f"Database ready: {DB_PATH.resolve()}")


setup_database()

In [ ]:
def upsert_index(records: list[dict]) -> int:
    """
    Upsert headline index records into the local SQLite database.

    INSERT OR REPLACE ensures that re-polling the same 2-hour window does not
    create duplicate rows. Records are uniquely identified by the composite key
    (ticker, publication_time, topic_name, index_type).

    Returns the number of rows written.
    """
    if not records:
        return 0

    columns = [
        "ticker",
        "publication_time",
        "topic_name",
        "index_type",
        "headline_count",
        "sentiment_sum",
        "sentiment_abs_sum",
        "sentiment_std",
        "fetched_at",
    ]
    placeholders = ", ".join(["?"] * len(columns))
    sql = f"INSERT OR REPLACE INTO headline_index ({', '.join(columns)}) " f"VALUES ({placeholders})"
    rows = [tuple(r.get(c) for c in columns) for r in records]

    with get_connection() as conn:
        conn.executemany(sql, rows)
    return len(rows)


print("upsert_index() defined")

## Dry Run

Fetch and store index data for all configured tickers once. Use this to verify your API key and inspect the raw schema before starting the polling loop.

In [ ]:
print("Running dry-run poll for all tickers...\n")
dry_run_rows = []

for ticker in TICKERS:
    try:
        records = fetch_live_index(ticker)
        n_written = upsert_index(records)
        dry_run_rows.append({"ticker": ticker, "records_fetched": len(records), "rows_written": n_written})
        print(f"  {ticker:12s}: {len(records):4d} index records | {n_written:4d} rows written")
    except requests.HTTPError as e:
        print(f"  {ticker}: HTTP {e.response.status_code} — {e.response.text[:200]}")
    except Exception as e:
        print(f"  {ticker}: {type(e).__name__}: {e}")

print()
display(pd.DataFrame(dry_run_rows))

# Preview stored data
with get_connection() as conn:
    sample = pd.read_sql(
        "SELECT * FROM headline_index ORDER BY publication_time DESC LIMIT 10",
        conn,
    )
print("\nSample rows from database:")
display(sample)

## Initial Visualisation

Inspect the first hourly snapshot: net sentiment by bucket and headline volume (news flow) per ticker.

In [ ]:
with get_connection() as conn:
    df = pd.read_sql("SELECT * FROM headline_index", conn)

df["publication_time"] = pd.to_datetime(df["publication_time"], utc=True)
df["hour_label"] = df["publication_time"].dt.strftime("%H:%M")

# Derive conviction ratio (net directional strength)
df["conviction_ratio"] = df["sentiment_sum"] / df["sentiment_abs_sum"].replace(0, float("nan"))

if df.empty:
    print("No data yet — run the dry-run cell above first.")
else:
    active_tickers = [t for t in TICKERS if t in df["ticker"].unique()]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # ── Plot 1: Net sentiment_sum per ticker (grouped bar by hour) ─────────
    ax = axes[0]
    hourly = df.groupby(["ticker", "publication_time"])["sentiment_sum"].sum().reset_index()
    colors = sns.color_palette("muted", len(active_tickers))
    for i, ticker in enumerate(active_tickers):
        sub = hourly[hourly["ticker"] == ticker].sort_values("publication_time")
        ax.bar(
            sub["publication_time"].dt.strftime("%H:%M"),
            sub["sentiment_sum"],
            alpha=0.75,
            label=ticker,
            color=colors[i],
        )
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title("Net Sentiment Sum by Hour", fontsize=11, fontweight="bold")
    ax.set_xlabel("Hour (UTC)")
    ax.set_ylabel("Sentiment Sum")
    ax.legend(fontsize=8)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right", fontsize=8)

    # ── Plot 2: Headline count (news flow) per ticker over the window ──────
    ax = axes[1]
    volume = df.groupby(["ticker", "publication_time"])["headline_count"].sum().reset_index()
    for i, ticker in enumerate(active_tickers):
        sub = volume[volume["ticker"] == ticker].sort_values("publication_time")
        ax.plot(
            sub["publication_time"].dt.strftime("%H:%M"),
            sub["headline_count"],
            marker="o",
            linewidth=1.6,
            label=ticker,
            color=colors[i],
        )
    ax.set_title("Headline Count (News Flow) by Hour", fontsize=11, fontweight="bold")
    ax.set_xlabel("Hour (UTC)")
    ax.set_ylabel("Headline Count")
    ax.legend(fontsize=8)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right", fontsize=8)

    plt.suptitle(
        f"Initial Index Snapshot  —  {df['publication_time'].max().strftime('%Y-%m-%d %H:%M UTC')}",
        fontsize=12,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

    print(f"Total rows in database : {len(df)}")
    print(df.groupby("ticker")[["headline_count", "sentiment_sum", "sentiment_abs_sum"]].sum().round(4))

## 15-Minute Polling Loop

This cell runs indefinitely, polling all configured tickers every 15 minutes and upserting results into the local SQLite database.

**Stop:** press the ■ (Interrupt Kernel) button or use **Kernel → Interrupt**.

In production this logic would run as a scheduled job (cron, Airflow, or a Docker Compose service). This notebook form is suitable for research and supervised live monitoring.

In [ ]:
poll_count = 0
print(f"Starting polling loop — {len(TICKERS)} tickers every {POLL_INTERVAL_SECONDS // 60} min")
print("Interrupt the kernel to stop gracefully.\n")

try:
    while True:
        poll_count += 1
        t_start = datetime.now(timezone.utc)
        print(f"[Poll {poll_count}]  {t_start.strftime('%Y-%m-%d %H:%M:%S UTC')}")

        for ticker in TICKERS:
            try:
                records = fetch_live_index(ticker)
                n = upsert_index(records)
                print(f"  {ticker:12s}: {len(records):4d} records | {n:4d} upserted")
            except requests.HTTPError as e:
                print(f"  {ticker}: HTTP {e.response.status_code} — skipping, will retry next poll")
            except Exception as e:
                print(f"  {ticker}: {type(e).__name__}: {e}")

        elapsed = (datetime.now(timezone.utc) - t_start).total_seconds()
        wait = max(0.0, POLL_INTERVAL_SECONDS - elapsed)
        print(f"  Completed in {elapsed:.1f}s.  Next poll in {wait / 60:.1f} min.\n")
        time.sleep(wait)

except KeyboardInterrupt:
    print(f"\nPolling stopped after {poll_count} poll(s).")
    print(f"All data saved to: {DB_PATH.resolve()}")

## Post-Poll Visualisation

Run this cell after one or more polling cycles to explore the accumulated data:

1. `sentiment_sum` over time per ticker with ±1 std deviation band
2. Multi-ticker conviction ratio comparison (`sentiment_sum / sentiment_abs_sum`)
3. Headline count heatmap — ticker × hour

In [ ]:
with get_connection() as conn:
    df = pd.read_sql("SELECT * FROM headline_index", conn)

df["publication_time"] = pd.to_datetime(df["publication_time"], utc=True)
df["conviction_ratio"] = df["sentiment_sum"] / df["sentiment_abs_sum"].replace(0, float("nan"))
df = df.sort_values("publication_time")

if df.empty:
    print("No data yet — run the polling loop above first.")
else:
    active_tickers = [t for t in TICKERS if t in df["ticker"].unique()]
    colors = sns.color_palette("muted", len(active_tickers))

    # Aggregate across topics to get one (ticker, time) row for plotting
    ts = (
        df.groupby(["ticker", "publication_time"])
        .agg(
            sentiment_sum=("sentiment_sum", "sum"),
            sentiment_abs_sum=("sentiment_abs_sum", "sum"),
            sentiment_std=("sentiment_std", "mean"),
            headline_count=("headline_count", "sum"),
        )
        .reset_index()
    )
    ts["conviction_ratio"] = ts["sentiment_sum"] / ts["sentiment_abs_sum"].replace(0, float("nan"))

    # ── Plot 1: Sentiment sum with std deviation band per ticker ──────────
    fig, axes = plt.subplots(
        len(active_tickers),
        1,
        figsize=(14, 3.0 * len(active_tickers)),
        sharex=True,
    )
    if len(active_tickers) == 1:
        axes = [axes]

    for ax, (ticker, color) in zip(axes, zip(active_tickers, colors)):
        sub = ts[ts["ticker"] == ticker].sort_values("publication_time")
        if sub.empty:
            ax.set_title(f"{ticker}  — no data")
            continue
        ax.plot(
            sub["publication_time"],
            sub["sentiment_sum"],
            color=color,
            linewidth=1.8,
            label="Sentiment sum",
        )
        ax.fill_between(
            sub["publication_time"],
            sub["sentiment_sum"] - sub["sentiment_std"],
            sub["sentiment_sum"] + sub["sentiment_std"],
            alpha=0.18,
            color=color,
            label="±1 std dev",
        )
        ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
        ax.set_ylabel("Sentiment Sum", fontsize=9)
        ax.set_title(ticker, fontsize=10, fontweight="bold")
        ax.legend(fontsize=8, loc="upper left")
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    plt.suptitle("Sentiment Sum Over Time  (±1 Std Dev)", fontsize=13, fontweight="bold")
    plt.xlabel("Time (UTC)")
    plt.tight_layout()
    plt.show()

    # ── Plot 2: Multi-ticker conviction ratio comparison ───────────────────
    fig, ax = plt.subplots(figsize=(14, 4))
    for ticker, color in zip(active_tickers, colors):
        sub = ts[ts["ticker"] == ticker].sort_values("publication_time")
        ax.plot(
            sub["publication_time"],
            sub["conviction_ratio"],
            color=color,
            linewidth=1.8,
            label=ticker,
            marker="o",
            markersize=4,
        )
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.fill_between(
        ts["publication_time"].unique(),
        -0.1,
        0.1,
        alpha=0.06,
        color="grey",
        label="Neutral band (±0.1)",
    )
    ax.set_ylim(-1.05, 1.05)
    ax.set_title("Conviction Ratio by Ticker  (sentiment_sum / sentiment_abs_sum)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Time (UTC)")
    ax.set_ylabel("Conviction Ratio  [−1, +1]")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # ── Plot 3: Headline count heatmap — ticker × hour ─────────────────────
    heatmap_df = (
        df.groupby(["ticker", df["publication_time"].dt.strftime("%H:%M")])["headline_count"]
        .sum()
        .unstack("publication_time")
        .fillna(0)
    )
    if not heatmap_df.empty:
        fig, ax = plt.subplots(figsize=(14, max(2.5, len(active_tickers) * 1.4)))
        sns.heatmap(
            heatmap_df,
            ax=ax,
            cmap="YlOrRd",
            linewidths=0.4,
            annot=True,
            fmt=".0f",
            cbar_kws={"label": "Headline Count"},
        )
        ax.set_title("Headline Count Heatmap  —  Ticker × Hour", fontsize=13, fontweight="bold")
        ax.set_xlabel("Hour (UTC)")
        ax.set_ylabel("Ticker")
        plt.xticks(rotation=30, ha="right", fontsize=8)
        plt.tight_layout()
        plt.show()

## Strategy Usage Example

Demonstrates how to load the accumulated index from the local database and derive a normalised systematic signal.

**Signal logic:** aggregate topic-level index records to (ticker, hour) level, compute the conviction ratio, smooth with a 2-period rolling mean, then threshold to **LONG / SHORT / FLAT**.

| Metric | Formula | Range |
|---|---|---|
| `conviction_ratio` | `sentiment_sum / sentiment_abs_sum` | [−1, +1] |
| `headline_count` | sum of headlines per bucket | ≥ 0 |
| `sentiment_std` | mean std dev across topics | ≥ 0 |

In [ ]:
with get_connection() as conn:
    df = pd.read_sql("SELECT * FROM headline_index", conn)

df["publication_time"] = pd.to_datetime(df["publication_time"], utc=True)

# ── 1. Aggregate to (ticker, hour) ─────────────────────────────────────────────
df_agg = (
    df.groupby(["ticker", "publication_time"])
    .agg(
        sentiment_sum=("sentiment_sum", "sum"),
        sentiment_abs_sum=("sentiment_abs_sum", "sum"),
        headline_count=("headline_count", "sum"),
        sentiment_std=("sentiment_std", "mean"),
    )
    .reset_index()
    .sort_values(["ticker", "publication_time"])
)

# ── 2. Compute conviction ratio ────────────────────────────────────────────────
df_agg["conviction_ratio"] = df_agg["sentiment_sum"] / df_agg["sentiment_abs_sum"].replace(0, float("nan"))

# ── 3. Smooth with a 2-period (2-hour) rolling mean ───────────────────────────
df_agg["signal_smooth"] = df_agg.groupby("ticker")["conviction_ratio"].transform(
    lambda x: x.rolling(2, min_periods=1).mean()
)

# ── 4. Threshold-based position signal ────────────────────────────────────────
BULLISH_THRESHOLD = 0.10  # above this → LONG
BEARISH_THRESHOLD = -0.10  # below this → SHORT

df_agg["signal"] = "FLAT"
df_agg.loc[df_agg["signal_smooth"] >= BULLISH_THRESHOLD, "signal"] = "LONG"
df_agg.loc[df_agg["signal_smooth"] <= BEARISH_THRESHOLD, "signal"] = "SHORT"

# ── 5. Latest signal per ticker ────────────────────────────────────────────────
latest = (
    df_agg.sort_values("publication_time")
    .groupby("ticker")
    .last()
    .reset_index()[
        [
            "ticker",
            "publication_time",
            "conviction_ratio",
            "headline_count",
            "sentiment_std",
            "signal",
        ]
    ]
)
print("Latest strategy signals:")
display(latest)

# ── 6. Signal history chart ────────────────────────────────────────────────────
if len(df_agg) > 1:
    colors = sns.color_palette("muted", len(df_agg["ticker"].unique()))
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    # Conviction ratio + signal thresholds
    ax = axes[0]
    for ticker, color in zip(df_agg["ticker"].unique(), colors):
        grp = df_agg[df_agg["ticker"] == ticker]
        ax.plot(grp["publication_time"], grp["signal_smooth"], label=ticker, linewidth=1.8, color=color)
    ax.axhline(BULLISH_THRESHOLD, color="#2ecc71", linewidth=1.0, linestyle="--", label="Bullish threshold")
    ax.axhline(BEARISH_THRESHOLD, color="#e74c3c", linewidth=1.0, linestyle="--", label="Bearish threshold")
    ax.axhline(0, color="black", linewidth=0.6, linestyle=":")
    ax.set_ylim(-1.05, 1.05)
    ax.set_title("Smoothed Conviction Ratio Signal by Ticker", fontsize=11, fontweight="bold")
    ax.set_ylabel("Conviction Ratio  [−1, +1]")
    ax.legend(fontsize=8)

    # Headline count (news flow)
    ax = axes[1]
    for ticker, color in zip(df_agg["ticker"].unique(), colors):
        grp = df_agg[df_agg["ticker"] == ticker]
        ax.bar(
            grp["publication_time"],
            grp["headline_count"],
            label=ticker,
            color=color,
            alpha=0.65,
            width=0.03,
        )
    ax.set_title("Headline Count (News Flow) — Context for Signal", fontsize=11, fontweight="bold")
    ax.set_xlabel("Time (UTC)")
    ax.set_ylabel("Headline Count")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

## Historical Backfill

### Why You Need This

The live endpoint (`/v1/headlines/index/ticker/live/{ticker}`) returns only the **most recent 2-hour window**. As a result, the local database only contains data from when polling started — charts have no historical depth and are unsuitable for strategy calibration or internal dashboards.

The **historical endpoint** (`/v1/headlines/index/ticker/historical/{ticker}`) solves this:

| Property | Live | Historical |
|---|---|---|
| Window | Last 2 hours | Up to **90 days** back |
| Pagination | None (single response) | Keyset via `next_token` |
| Response schema | `list[HeadlineIndex]` | `PaginatedHeadlineIndexResponse` (`data`, `has_more`, `next_token`) |

### Backfill Strategy

Because both endpoints write into the **same SQLite table** via `upsert_index()`, backfilled records automatically merge with live-polled records. No schema changes or duplicate records are introduced.

**Pagination pattern:** each response contains `has_more` and `next_token`. Pass `next_token` as a query parameter on the next request and repeat until `has_more` is `False`.

```
fetch page 1 (start_date) ──► upsert ──► has_more?
                                              │
                              Yes ◄───────────┘──► fetch page N+1 (next_token) ──► upsert
                              No  ──► done
```

In [ ]:
from datetime import timedelta

BACKFILL_DAYS = 7  # days of history to fetch; max 90


def fetch_historical_index(ticker: str, start_date, end_date=None) -> list[dict]:
    """
    Paginate through GET /v1/headlines/index/ticker/historical/{ticker} and return
    all matching records as a flat list.

    Keyset pagination: each response includes 'has_more' and 'next_token'.
    We loop until has_more is False, passing next_token into every subsequent request.
    """
    url = f"{BASE_URL}/headlines/index/ticker/historical/{ticker}"
    params = {
        "start_date": start_date.isoformat(),
        "index_type": INDEX_TYPE,
        "topic_preset": TOPIC_PRESET,
        "sparse": str(SPARSE).lower(),
        "align_to_period_end": str(ALIGN_TO_PERIOD_END).lower(),
        "limit": 1000,
    }
    if end_date:
        params["end_date"] = end_date.isoformat()
    headers = {"x-api-key": API_KEY}

    all_records, page = [], 1
    while True:
        response = requests.get(url, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        body = response.json()
        records = body["data"]
        for r in records:
            r["fetched_at"] = datetime.now(timezone.utc).isoformat()
        all_records.extend(records)
        print(f"    page {page:3d}: {len(records):5d} records  |  total so far: {len(all_records):6d}")
        if not body["has_more"]:
            break
        params["next_token"] = body["next_token"]
        page += 1
    return all_records


def backfill_all_tickers(days_back: int = BACKFILL_DAYS) -> None:
    """Fetch and upsert historical headline index for all configured tickers."""
    start = (datetime.utcnow() - timedelta(days=days_back)).date()
    print(f"Backfilling {len(TICKERS)} tickers from {start} ({days_back} days)\n")
    for ticker in TICKERS:
        print(f"  {ticker}")
        try:
            records = fetch_historical_index(ticker, start)
            n = upsert_index(records)
            print(f"    {n} rows upserted into headline_index\n")
        except requests.HTTPError as e:
            print(f"    HTTP {e.response.status_code} — skipping\n")
        except Exception as e:
            print(f"    {type(e).__name__}: {e}\n")


backfill_all_tickers()

In [ ]:
# Load the full DB — now includes both backfilled history and live-polled data
with get_connection() as conn:
    df = pd.read_sql("SELECT * FROM headline_index", conn)

df["publication_time"] = pd.to_datetime(df["publication_time"], utc=True)
df["date"] = df["publication_time"].dt.date
df["conviction_ratio"] = df["sentiment_sum"] / df["sentiment_abs_sum"].replace(0, float("nan"))

if df.empty:
    print("No data — run the backfill cell above first.")
else:
    active_tickers = [t for t in TICKERS if t in df["ticker"].unique()]
    colors = sns.color_palette("muted", len(active_tickers))
    date_range = sorted(df["date"].unique())

    # Aggregate to (ticker, date) — sum across topics
    daily = (
        df.groupby(["ticker", "date"])
        .agg(
            sentiment_sum=("sentiment_sum", "sum"),
            sentiment_abs_sum=("sentiment_abs_sum", "sum"),
            headline_count=("headline_count", "sum"),
        )
        .reset_index()
    )
    daily["conviction_ratio"] = daily["sentiment_sum"] / daily["sentiment_abs_sum"].replace(0, float("nan"))

    # ── Plot 1: Daily conviction ratio per ticker ──────────────────────────
    fig, ax = plt.subplots(figsize=(14, 5))
    for ticker, color in zip(active_tickers, colors):
        sub = daily[daily["ticker"] == ticker].sort_values("date")
        ax.plot(
            sub["date"], sub["conviction_ratio"], marker="o", linewidth=1.8, markersize=5, label=ticker, color=color
        )
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_ylim(-1.05, 1.05)
    ax.set_title(
        "Daily Conviction Ratio by Ticker  (sentiment_sum / sentiment_abs_sum)", fontsize=13, fontweight="bold"
    )
    ax.set_xlabel("Date")
    ax.set_ylabel("Conviction Ratio  [−1, +1]")
    ax.legend(fontsize=9)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    # ── Plot 2: Headline count heatmap — ticker × date ─────────────────────
    heatmap_df = daily.groupby(["ticker", "date"])["headline_count"].sum().unstack("date").fillna(0)
    # Limit x-axis to last 30 days for readability
    heatmap_df = heatmap_df[sorted(heatmap_df.columns)[-30:]]

    fig, ax = plt.subplots(figsize=(16, max(2.5, len(active_tickers) * 1.4)))
    sns.heatmap(
        heatmap_df,
        ax=ax,
        cmap="YlOrRd",
        linewidths=0.3,
        annot=len(date_range) <= 14,
        fmt=".0f",
        cbar_kws={"label": "Headline Count"},
    )
    ax.set_title("Daily Headline Count Heatmap  —  Ticker × Date", fontsize=13, fontweight="bold")
    ax.set_xlabel("Date")
    ax.set_ylabel("Ticker")
    plt.xticks(rotation=40, ha="right", fontsize=8)
    plt.tight_layout()
    plt.show()

    print(f"Date range in DB: {min(date_range)} → {max(date_range)}  ({len(date_range)} days)")
    print(f"Total rows      : {len(df):,}")

## Next Steps: Production Deployment

This notebook is an **integration reference** — it covers the complete workflow from API authentication through to a strategy-ready normalised signal in a single, runnable document.

For a production deployment, see the accompanying **Polling Application** in `app/live_polling`, which packages this workflow into:

- A standalone **polling service** with configurable intervals and ticker lists
- A **PostgreSQL** store replacing the local SQLite database for multi-process access and durability
- A lightweight **REST API** exposing the latest conviction signals to downstream order management systems
- A **monitoring dashboard** for real-time observation of news flow, sentiment drift, and signal transitions

### Choosing Between the Index and Raw Feed

- Use the **index endpoint** (this notebook) when you want a ready-to-use hourly signal with minimal client-side processing.
- Use the **raw feed endpoint** (`notebooks/live/headline_sentiment_polling.ipynb`) when you need per-headline granularity, custom aggregation logic, or are building NLP features for a machine learning model.

### Related Resources

- [Permutable AI API Documentation](https://docs.permutable.ai)
- `notebooks/backtesting/index_cross_ticker_signal_assessment.ipynb` — full backtesting pipeline for validating the index signal on historical data
- `notebooks/live/headline_sentiment_polling.ipynb` — equivalent guide using the per-headline raw feed endpoint